# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Pick a baseline and up to 4 variant scenarios below — every chart and table in
this notebook updates automatically as the selection changes, no re-running
required. Each must already have a run (`analysis sim load <name> && analysis
sim run <name>`); the newest run for each is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).



In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.connection import RunConnection
from analysis.sim.notebook_ui.pickers import dollar_plot_mode, multi_scenario_picker
from analysis.sim.notebook_ui.compare_plots import (
    comparability_widget, rules_and_events_widget,
    net_worth_compare_widget, breakeven_widget,
    year_end_widget,
)

MAX_VARIANTS = 4

# CONNECTIONS holds the live RunConnections, refreshed in place whenever either picker changes.
PICKER, CONNECTIONS = multi_scenario_picker(default_baseline="current", max_variants=MAX_VARIANTS)
PLOT_MODE = dollar_plot_mode()
PICKER

## Are these runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
CONNECTIONS.show(comparability_widget)

## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
CONNECTIONS.show(rules_and_events_widget)

In [ ]:
CONNECTIONS.show(rules_and_events_widget)

## Net worth over time

Use the shared **Dollar basis** toggle to switch between nominal and CPI-constant dollars. The selected basis applies to the net-worth chart and the comparison tables below.

In [ ]:
CONNECTIONS.show(net_worth_compare_widget, plot_mode=PLOT_MODE)

### Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
CONNECTIONS.show(breakeven_widget, plot_mode=PLOT_MODE)

## Account Comparisons

In [ ]:
from collections.abc import Sequence
from analysis.sim.connection import RunConnection
from analysis.sim.notebook_ui.pickers import MultiScenarioConnections
import ipywidgets as widgets
import pandas as pd


def compare_accounts(runs: MultiScenarioConnections) -> widgets.VBox:
    """
    Compare the accounts across multiple scenario connections.

    Parameters
    ----------
    runs : MultiScenarioConnections
        The multi-scenario connections object containing the scenarios to compare.

    Returns
    -------
    widgets.VBox
        The VBox widget containing the comparison display.
    """
    return widgets.VBox()

CONNECTIONS.show(compare_accounts)
v = [
        CONNECTIONS.connections[variant] for variant in CONNECTIONS.variants
    ]
base = CONNECTIONS.connections[CONNECTIONS.baseline]

def group_account_balances(base: RunConnection) -> dict[str, float]:
    """
    Get the account balances grouped by account.

    Parameters
    ----------
    base : RunConnection
        The run connection for the baseline scenario.

    Returns
    -------
    dict[str, float]
        A dictionary mapping account names to their monthly balances.
    """
    balances = base.items("account_balance").copy()
    balances['month'] = pd.to_datetime(balances['month']).dt.strftime("%Y-%m")

    return balances.pivot(index="month", columns="account", values="balance")
def compare_account_balance_histories(_, baseline_rc: RunConnection,
                                      variants_rcs: Sequence[RunConnection],
                                      **kwargs) -> None:
    ...

In [ ]:

def p(v):
    print(f"{v=}")
    return 

#bg['Business']
def compare_account_balances(_, baseline_rc: RunConnection,
                             variant_rcs: Sequence[RunConnection],
                             **kwargs) -> None:
    """
    Compare the account balances across multiple scenario connections.

    Parameters
    ----------
    _ : Any
        Placeholder for the first argument.
    baseline_rc : RunConnection
        The run connection for the baseline scenario.
    variants_rcs : Sequence[RunConnection]
        The run connections for the variant scenarios.
    **kwargs : dict
        Additional keyword arguments.

    Returns
    -------
    None
    """
    bg = group_account_balances(baseline_rc)
    vgs = {
        name: group_account_balances(variant)
        for name, variant in variant_rcs.items()
        if variant is not baseline_rc}
    bg_keys = set(bg.keys())
    vgs_keys = {
        variant: set(groups.keys()) for variant, groups in vgs.items()
    }
    added = {variant: keys - bg_keys for variant, keys in vgs_keys.items()}
    deleted = {variant: bg_keys - keys for variant, keys in vgs_keys.items()}
    diff = {variant: vgs_keys[variant] ^ bg_keys for variant in vgs_keys.keys()}
    return (added, deleted, diff)

CONNECTIONS.show(compare_account_balances)

### Annual net worth comparisons and deltas.|

In [ ]:
CONNECTIONS.show(year_end_widget, plot_mode=PLOT_MODE)